# 2.0 · Exploratory data analysis of MSLesSeg

Goal: understand the dataset before training any model: size, patients, lesion load, lesion sizes, class imbalance, image intensities and scanners.

Per-study statistics are computed by `ms_seg/eda.py` (lesions = connected components of the mask, **6-connectivity**, which reproduces the authors' lesion counts exactly; see section 3.3) and cached in `data/interim/`. Figures are saved to `reports/figures/`.

Related issue: #3

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ms_seg import eda
from ms_seg.config import FIGURES_DIR, INTERIM_DATA_DIR

# Compute (or reuse) the cached statistics. First run takes a few minutes.
if not (INTERIM_DATA_DIR / "study_stats.csv").exists():
    eda.compute_all()

studies = pd.read_csv(INTERIM_DATA_DIR / "study_stats.csv")
lesions = pd.read_csv(INTERIM_DATA_DIR / "lesions.csv")
clinical = eda.load_clinical()
scanners = eda.load_scanners(modality="FLAIR")
df = (studies.merge(clinical, on=["patient", "timepoint"], how="left")
             .merge(scanners, on=["patient", "timepoint"], how="left"))

# Plot style (thesis figures)
BLUE, ORANGE = "#2a78d6", "#eb6834"          # train, test
SPLIT_COLORS = {"train": BLUE, "test": ORANGE}
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#8a8984", "axes.labelcolor": "#0b0b0b", "text.color": "#0b0b0b",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
})
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
def save(fig, name):
    fig.savefig(FIGURES_DIR / f"eda_{name}.png")
df.shape, lesions.shape

2026-09-24 21:42:26.091 | INFO     | ms_seg.config:<module>:11 - PROJ_ROOT path is: /sessions/rcw-01jfmtx7ebsjpvekrgaqwziz/mnt/Esclerosi-Multiple


((115, 31), (3595, 4))

## 1. Dataset size

In [2]:
overview = df.groupby("split").agg(patients=("patient", "nunique"), studies=("study_id", "count"),
                                   lesions=("n_lesions", "sum")).loc[["train", "test"]]
overview.loc["total"] = overview.sum()
overview

,patients,studies,lesions
split,,,
train,53.0,93.0,2688.0
test,22.0,22.0,907.0
total,75.0,115.0,3595.0


In [3]:
tp_per_patient = df[df.split == "train"].groupby("patient").timepoint.nunique().value_counts().sort_index()
tp_per_patient.rename_axis("timepoints per patient").rename("train patients").to_frame().T

timepoints per patient,1,2,3,4
train patients,28,15,5,5


## 2. Patients (demographics)

In [4]:
patients = clinical.groupby("patient").first().join(df.groupby("patient").split.first())
demo = patients.groupby("split").agg(
    n=("age", "size"), age_mean=("age", "mean"), age_sd=("age", "std"),
    female=("sex", lambda s: (s == "F").sum()), male=("sex", lambda s: (s == "M").sum()),
    RRMS=("ms_type", lambda s: (s == "RRMS").sum()), SPMS=("ms_type", lambda s: (s == "SPMS").sum()),
    PPMS=("ms_type", lambda s: (s == "PPMS").sum()), EDSS_median=("edss", "median"),
).loc[["train", "test"]].round(1)
demo

,n,age_mean,age_sd,female,male,RRMS,SPMS,PPMS,EDSS_median
split,,,,,,,,,
train,53,36.4,10.2,32,21,50,3,0,2.0
test,22,38.5,10.5,16,6,21,0,1,1.0


## 3. Sanity checks

In [5]:
checks = {
    "studies with an empty mask": int((df.n_lesions == 0).sum()),
    "lesion volume == volume reported by the authors": f"{(df.lesion_voxels == df.reported_lesion_volume).mean():.0%}",
    "mean difference in lesion count vs. reported (ours - theirs)": round((df.n_lesions - df.reported_lesion_number).mean(), 2),
}
for mod in eda.MODALITIES:
    checks[f"studies with an empty {mod} (99th percentile = 0)"] = df.loc[df[f"{mod}_p99"] <= 0, "study_id"].tolist()
checks

{'studies with an empty mask': 0,
 'lesion volume == volume reported by the authors': '100%',
 'mean difference in lesion count vs. reported (ours - theirs)': np.float64(0.0),
 'studies with an empty FLAIR (99th percentile = 0)': [],
 'studies with an empty T1 (99th percentile = 0)': ['P49_T2'],
 'studies with an empty T2 (99th percentile = 0)': []}

Lesion volumes and lesion counts match those reported by the authors exactly (see section 3.3 for the counting rule).

**⚠ Check the studies listed with an empty modality** (see figure below) before training.

In [6]:
bad = df.loc[df["T1_p99"] <= 0, "study_id"].tolist()
if bad:
    sid = bad[0]
    row = df.set_index("study_id").loc[sid]
    sdir = eda.DATA_DIR / row.split / row.patient / (row.timepoint if row.split == "train" else "")
    vols = {m: eda.load_volume(sdir, m) for m in ["FLAIR", "T1", "T2"]}
    z = vols["FLAIR"].shape[2] // 2 + 10
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    for ax, (m, v) in zip(axes, vols.items()):
        ax.imshow(v[:, :, z].T, cmap="gray", origin="lower"); ax.set_title(m); ax.axis("off")
    fig.suptitle(f"{sid}: check the T1 volume", fontweight="bold", y=1.0)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    save(fig, "check_empty_t1")

### 3.1 Registration check: brain volume

All images should be registered to the same MNI152 template, so the number of brain voxels should be similar across studies. Studies whose brain is much larger than the rest suggest a registration problem (brain not scaled to the template and cropped at the field-of-view border).

In [7]:
med = df.brain_voxels.median()
outl = df.loc[df.brain_voxels > 1.4 * med, ["study_id", "split", "brain_voxels"]].assign(ratio_to_median=lambda d: (d.brain_voxels / med).round(2))
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(df.brain_voxels / 1e6, bins=40, color=BLUE, edgecolor="white", linewidth=0.8)
ax.axvline(1.4 * med / 1e6, color="#52514e", linestyle="--", linewidth=1)
ax.text(1.4 * med / 1e6 * 1.01, ax.get_ylim()[1] * 0.9, "1.4 × median", color="#52514e", fontsize=9)
ax.set_xlabel("brain volume (million voxels; 1 voxel = 1 mm³)"); ax.set_ylabel("studies")
ax.set_title("Brain volume after registration to MNI152")
fig.tight_layout(); save(fig, "brain_volume")
outl

,study_id,split,brain_voxels,ratio_to_median
66,P44_T1,train,3295440.0,1.73
67,P45_T1,train,3493134.0,1.84
81,P53_T1,train,3331450.0,1.75


In [8]:
if len(outl):
    ref = df.loc[(df.brain_voxels - med).abs().idxmin(), "study_id"]
    show = [ref] + outl.study_id.tolist()
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 2.9 * len(show)))
    for r, sid in enumerate(show):
        row = df.set_index("study_id").loc[sid]
        sdir = eda.DATA_DIR / row.split / row.patient / (row.timepoint if row.split == "train" else "")
        v = eda.load_volume(sdir, "FLAIR")
        vmax = np.percentile(v[v > 0], 99)
        axes[r, 0].imshow(v[v.shape[0] // 2].T, cmap="gray", origin="lower", vmax=vmax)
        axes[r, 1].imshow(v[:, :, v.shape[2] // 2].T, cmap="gray", origin="lower", vmax=vmax)
        label = "reference (median)" if sid == ref else f"{row.brain_voxels / med:.2f} × median"
        axes[r, 0].set_title(f"{sid}: {label}", loc="left", fontsize=10)
        for ax in axes[r]: ax.axis("off")
    fig.tight_layout(); save(fig, "check_registration")

### 3.2 Are the flagged studies usable?

Check that each flagged study's mask is consistent with its own images: no lesion voxel outside the brain, and lesions hyperintense on FLAIR (ratio lesion/brain median intensity similar to the other studies).

In [9]:
df["flair_contrast"] = df.FLAIR_lesion_median / df.FLAIR_p50
flagged = ["P49_T2", "P44_T1", "P45_T1", "P53_T1"]
rows = []
for sid in flagged:
    row = df.set_index("study_id").loc[sid]
    sdir = eda.DATA_DIR / row.split / row.patient / row.timepoint
    mask = eda.load_volume(sdir, "MASK") > 0.5
    flair = eda.load_volume(sdir, "FLAIR")
    rows.append({"study_id": sid, "mask_voxels": int(mask.sum()), "mask_outside_brain": int((mask & (flair == 0)).sum()),
                 "flair_contrast": round(row.flair_contrast, 2)})
print(f"FLAIR contrast in all studies: median {df.flair_contrast.median():.2f}, min {df.flair_contrast.min():.2f}")
pd.DataFrame(rows)

FLAIR contrast in all studies: median 1.45, min 1.12


,study_id,mask_voxels,mask_outside_brain,flair_contrast
0,P49_T2,7763,0,1.29
1,P44_T1,3970,0,1.54
2,P45_T1,7390,0,1.43
3,P53_T1,4114,0,1.50


**Decision:** masks are aligned with their images, so the four studies are kept in the main experiments (as the reference study most likely did). A sensitivity analysis without them will be reported.

### 3.3 How lesions are counted: matching the authors

A lesion is a connected component of the mask, but the count depends on the connectivity rule (6: voxels share a face; 18: face or edge; 26: face, edge or corner) and on whether very small components are discarded. The paper does not state its rule, so we compare every combination with the lesion numbers released in `clinical_data.csv`.

In [10]:
conn = eda.compare_connectivity()
ref = clinical.assign(study_id=clinical.patient + "_" + clinical.timepoint)[["study_id", "reported_lesion_number"]]
cmp = conn.merge(ref, on="study_id")
rules = []
for col in [c for c in conn.columns if c.startswith("c")]:
    diff = cmp[col] - cmp.reported_lesion_number
    k, m = col[1:].split("_min")
    rules.append({"connectivity": int(k), "min size (voxels)": int(m), "exact match (% studies)": round(100 * (diff == 0).mean(), 1),
                  "mean abs. difference": round(diff.abs().mean(), 2),
                  "mean train": round(cmp.loc[cmp.split == "train", col].mean(), 1), "mean test": round(cmp.loc[cmp.split == "test", col].mean(), 1)})
rules = pd.DataFrame(rules).sort_values("mean abs. difference").reset_index(drop=True)
print("Paper (Table 1): mean 28.9 lesions (train), 41.2 (test)")
rules.head(8)

Paper (Table 1): mean 28.9 lesions (train), 41.2 (test)


,connectivity,min size (voxels),exact match (% studies),mean abs. difference,mean train,mean test
0,6,1,100.0,0.00,28.9,41.2
1,6,2,49.6,0.93,28.0,40.0
2,18,1,42.6,1.46,27.4,40.0
3,6,3,37.4,1.55,27.4,39.6
4,26,1,37.4,1.67,27.2,39.7
5,18,2,36.5,1.81,27.1,39.5
6,26,2,33.9,1.99,26.9,39.2
7,6,5,30.4,2.10,26.8,39.1


**6-connectivity without any size filter reproduces the authors' count in 100% of the studies** and the means reported in the paper. It is therefore used throughout this analysis and will be used for the lesion-wise metrics (LTPR, LFPR).

## 4. Lesion load per study

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
# train as filled bars, test as an outline on top (no colour mixing)
panels = [
    (axes[0], "lesion_voxels", np.logspace(np.log10(500), np.log10(80000), 25), "total lesion volume per study (mm³, log scale)", "Total lesion volume"),
    (axes[1], "n_lesions", np.arange(0, df.n_lesions.max() + 9, 8), "number of lesions per study", "Lesions per study"),
]
for ax, col, bins, xlabel, title in panels:
    tr, te = df[df.split == "train"], df[df.split == "test"]
    ax.hist(tr[col], bins=bins, color=BLUE, edgecolor="white", linewidth=0.8, label=f"train (n={len(tr)})")
    ax.hist(te[col], bins=bins, histtype="step", color=ORANGE, linewidth=2, label=f"test (n={len(te)})")
    ax.set_xlabel(xlabel); ax.set_ylabel("studies"); ax.set_title(title); ax.legend(frameon=False)
axes[0].set_xscale("log")
fig.tight_layout(); save(fig, "lesion_load")

In [12]:
df.groupby("split")[["lesion_voxels", "n_lesions", "lesion_fraction_pct"]].describe().T.round(2)

split                          test     train
lesion_voxels       count     22.00     93.00
                    mean   10231.55  12167.27
                    std    11169.30  13305.43
                    min      919.00    744.00
                    25%     3004.25   2857.00
                    50%     5601.50   6217.00
                    75%    11585.25  17877.00
                    max    42098.00  72872.00
n_lesions           count     22.00     93.00
                    mean      41.23     28.90
                    std       43.18     19.27
                    min        4.00      3.00
                    25%       15.00     16.00
                    50%       28.50     24.00
                    75%       49.75     39.00
                    max      193.00    111.00
lesion_fraction_pct count     22.00     93.00
                    mean       0.53      0.64
                    std        0.58      0.71
                    min        0.05      0.04
                    25%        0.16      0.14
                    50%        0.30      0.30
                    75%        0.59      0.94
                    max        2.25      3.85

## 5. Class imbalance

Fraction of brain voxels that are lesion. This is why plain accuracy is useless and why Dice-based losses are used.

In [13]:
frac = df.lesion_fraction_pct
print(f"Lesion voxels are {frac.median():.2f}% of the brain (median), range {frac.min():.2f}–{frac.max():.2f}%")
print(f"→ roughly 1 lesion voxel for every {int(100 / frac.median())} healthy brain voxels")

Lesion voxels are 0.30% of the brain (median), range 0.04–3.85%
→ roughly 1 lesion voxel for every 330 healthy brain voxels


## 6. Lesion size distribution

In [14]:
sizes = lesions.size_voxels
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.logspace(0, np.log10(sizes.max()), 40)
ax.hist(sizes, bins=bins, color=BLUE, edgecolor="white", linewidth=0.8)
ax.set_xscale("log")
for t, lab in [(10, "10 mm³"), (100, "100 mm³")]:
    ax.axvline(t, color="#52514e", linestyle="--", linewidth=1)
    ax.text(t * 1.08, ax.get_ylim()[1] * 0.92, lab, color="#52514e", fontsize=9)
ax.set_xlabel("lesion size (mm³ = voxels, log scale)"); ax.set_ylabel("lesions")
ax.set_title(f"Size of each lesion (n={len(sizes):,} lesions, all studies)")
fig.tight_layout(); save(fig, "lesion_sizes")

In [15]:
size_bins = pd.cut(sizes, [0, 10, 27, 100, 1000, np.inf],
                   labels=["≤10 mm³", "11–27 mm³", "28–100 mm³", "101–1000 mm³", ">1000 mm³"])
tab = pd.DataFrame({"lesions": size_bins.value_counts(sort=False)})
tab["% of lesions"] = (100 * tab.lesions / tab.lesions.sum()).round(1)
vol = sizes.groupby(size_bins, observed=False).sum()
tab["% of lesion volume"] = (100 * vol / vol.sum()).round(1)
tab

,lesions,% of lesions,% of lesion volume
size_voxels,,,
≤10 mm³,384,10.7,0.1
11–27 mm³,459,12.8,0.7
28–100 mm³,1520,42.3,6.4
101–1000 mm³,1054,29.3,21.3
>1000 mm³,178,5.0,71.5


**Key point for the thesis:** most lesions are small, but they represent a tiny part of the total lesion volume. A model can reach a good Dice while missing many small lesions, because Dice is dominated by the large ones. This motivates lesion-wise metrics (LTPR/LFPR) and the small-lesion work planned in M3.

## 7. Image intensities and scanners

Images are not intensity-normalised: each scanner/protocol gives a different range. Intensity normalisation (e.g. z-score per image, as nnU-Net does) will be needed.

In [16]:
scan_counts = df.scanner.fillna("unknown").value_counts()
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(scan_counts.index[::-1], scan_counts.values[::-1], color=BLUE, height=0.7)
for y, v in enumerate(scan_counts.values[::-1]):
    ax.text(v + 0.5, y, str(v), va="center", fontsize=9, color="#52514e")
ax.set_xlabel("studies (FLAIR)"); ax.set_title("Scanner model per study"); ax.grid(axis="y", visible=False)
fig.tight_layout(); save(fig, "scanners")

In [17]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=False)
groups = df.manufacturer.fillna("unknown")
order = groups.value_counts().index.tolist()
for ax, mod in zip(axes, eda.MODALITIES):
    data = [df.loc[groups == g, f"{mod}_p99"] for g in order]
    ax.boxplot(data, widths=0.5, patch_artist=True, showfliers=True,
               boxprops=dict(facecolor="#dbe8f8", edgecolor=BLUE), medianprops=dict(color=BLUE, linewidth=2),
               whiskerprops=dict(color="#8a8984"), capprops=dict(color="#8a8984"),
               flierprops=dict(marker="o", markersize=4, markerfacecolor=BLUE, markeredgecolor="white"))
    ax.set_xticks(range(1, len(order) + 1), [f"{g}\n(n={(groups == g).sum()})" for g in order])
    ax.set_title(f"{mod}: 99th percentile of brain intensity"); ax.set_ylabel("intensity (a.u.)")
fig.tight_layout(); save(fig, "intensities_by_manufacturer")

## 8. Longitudinal patients (train)

In [18]:
multi = df[df.split == "train"].groupby("patient").filter(lambda g: len(g) > 1)
fig, ax = plt.subplots(figsize=(8, 3.8))
for _, g in multi.groupby("patient"):
    g = g.sort_values("timepoint")
    ax.plot(g.timepoint, g.lesion_voxels, color=BLUE, alpha=0.45, linewidth=2, marker="o", markersize=5)
ax.set_yscale("log"); ax.set_xlabel("timepoint"); ax.set_ylabel("total lesion volume (mm³, log)")
ax.set_title(f"Lesion volume over time ({multi.patient.nunique()} patients with >1 timepoint)")
fig.tight_layout(); save(fig, "longitudinal")

In [19]:
# Patients whose lesion volume changes by more than a factor 3 between consecutive visits
# (possible real progression / regression, or an annotation or registration issue worth checking)
chg = (multi.sort_values(["patient", "timepoint"])
            .assign(prev=lambda d: d.groupby("patient").lesion_voxels.shift())
            .dropna(subset=["prev"]))
chg["ratio"] = chg.lesion_voxels / chg.prev
chg.loc[(chg.ratio > 3) | (chg.ratio < 1 / 3), ["patient", "timepoint", "prev", "lesion_voxels", "ratio"]].round(2)

,patient,timepoint,prev,lesion_voxels,ratio
30,P20,T2,744.0,5807.0,7.81
31,P20,T3,5807.0,1537.0,0.26
72,P49,T2,34709.0,7763.0,0.22
76,P50,T2,9144.0,36435.0,3.98


## 9. Summary

- **Size:** 75 patients, 115 studies, 3,595 lesions (6-connectivity). Train: 53 patients / 93 studies (1–4 timepoints). Test: 22 patients / 22 studies, **with masks**.
- **Patients:** mostly RRMS (71/75), 64% female, mean age ≈ 37. Train and test are similar.
- **Sanity checks:** no empty masks; lesion volumes **and** lesion counts match those reported by the authors in 100% of studies.
- **Counting rule:** the authors count lesions with **6-connectivity and no size filter** (exact match in 115/115 studies; means 28.9 train / 41.2 test as in the paper). We use the same rule for the EDA and for the lesion-wise metrics.
- **Data-quality flags:**
  - `P49_T2`: the **T1 volume is empty** and its lesion volume drops 4.5× vs. `P49_T1`.
  - `P44_T1`, `P45_T1`, `P53_T1`: brain volume **1.7–1.8× the median** → registration problem (brain not scaled to MNI, cropped at the border).
  - `P20` (T1→T2→T3) and `P50` (T1→T2): lesion volume changes >3× between visits.
  - The masks of the flagged studies are aligned with their images → **kept**, with a sensitivity analysis without them.
- **Class imbalance:** lesions are **0.30% of the brain** (median) → ~1 lesion voxel per 330 healthy voxels. Use Dice-based losses and lesion-aware patch sampling.
- **Small lesions:** **66% of lesions are ≤100 mm³ but they are only ~7% of the lesion volume**; lesions >1000 mm³ are 5% of lesions but ~72% of the volume. Voxel-wise Dice is dominated by large lesions → report lesion-wise metrics (LTPR, LFPR). This motivates the small-lesion work in M3.
- **Heterogeneity:** 9 scanner models (mostly Philips) and very different intensity ranges between manufacturers/protocols → per-image intensity normalisation (z-score), and a possible robustness analysis in M4.
- **Folds (#5):** split at **patient level** so that timepoints of the same patient never end up in different folds.
